# Spark Optimization

In [9]:
! java --version


openjdk 17.0.17 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


## Create Session

In [10]:
from pyspark.sql import SparkSession

In [11]:
spark=SparkSession.builder.master('local[*]')\
      .appName("SparkOptm")\
      .getOrCreate()
'''
builder: Initiates the builder pattern to construct the session.
master("local[*]"): Configures Spark to run locally using all available cores.
appName("My Spark Application"): Sets a name for the application, visible in the Spark web UI.
getOrCreate(): Returns the existing SparkSession if one is active, or creates a new one with the specified options if none exists.
'''

'\nbuilder: Initiates the builder pattern to construct the session.\nmaster("local[*]"): Configures Spark to run locally using all available cores.\nappName("My Spark Application"): Sets a name for the application, visible in the Spark web UI.\ngetOrCreate(): Returns the existing SparkSession if one is active, or creates a new one with the specified options if none exists.\n'

In [12]:
spark

## Turn AQE off

In [13]:
spark.conf.set("spark.sql.adaptive.enabled","false")

In [14]:
spark.conf.get("spark.sql.adaptive.enabled")

'false'

## Import dataset

In [16]:
df=spark.read.format("csv")\
.option("header",True)\
.option("inferSchema",True)\
.load('/content/BigMart Sales.csv')

In [15]:
from pyspark.sql.functions import *

In [17]:
df.limit(5).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

## Scanning Optimization

### Get Number of Partitions

In [18]:
df.rdd.getNumPartitions()

1

### Change default partition size to 128 KB

In [19]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

'134217728b'

In [20]:
str(int(134217728)/1024**2)+' MB'

'128.0 MB'

In [21]:
spark.conf.set("spark.sql.files.maxPartitionBytes",131072)

In [22]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

'131072'

#### Reread the data to check no of partitions

In [23]:
df=spark.read.format("csv")\
.option("header",True)\
.option("inferSchema",True)\
.load('/content/BigMart Sales.csv')

In [24]:
df.rdd.getNumPartitions()

7

In [25]:
spark.conf.set("spark.sql.files.maxPartitionBytes",134217728)

In [26]:
spark.conf.get("spark.sql.files.maxPartitionBytes")

'134217728'

### Repartition

In [27]:
df=df.repartition(10)

In [28]:
df.rdd.getNumPartitions()

10

In [29]:
df.withColumn("partion_id",spark_partition_id()).show()

+---------------+-----------+----------------+---------------+------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|         Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|partion_id|
+---------------+-----------+----------------+---------------+------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------+
|          DRG48|       5.78|         Low Fat|    0.014555066|       Soft Drinks|145.2102|           OUT046|                     1997|      Small|              Tier 1|Supermarket Type1|        3062.0142|         0|
|          FDE24|      14.85|         Low Fat|     0.09344495|      Baking Goods|141.0812|           OUT035|                     2004|      

### Write the partitioned data

In [30]:
! mkdir FileStore

In [31]:
df.write.format('parquet')\
.mode('append')\
.option("path","/content/FileStore")\
.save()

### Read partitioned Data

In [32]:
new_df=spark.read.format("csv")\
.option("header",True)\
.option("inferSchema",True)\
.load('/content/FileStore')

### PartitionBy

In [33]:
df.write.format("parquet")\
.mode("overwrite")\
.partitionBy("Outlet_Location_Type")\
.option("path","/content/FileStore")\
.save()
#less resources will be used when df is filtered say outlet_location_type == Tier 1

## Joins Optimization

In [34]:
spark.conf.get("spark.sql.adaptive.enabled")

'false'

In [46]:
transactions_df=spark.createDataFrame(
    [(1,"US",100),
     (2,"UK",200),
     (3,"NP",300),
     (4,"GER",400)],
    ["id","Country_Code","Amount"]
)
countries_df=spark.createDataFrame(
    [("US","United States of America"),
     ("UK","United Kingdom"),
     ("NP","Nepal"),
     ("GER","Germany")],
      ["Country_Code","Country_Name"]
    )

In [47]:
transactions_df.show()#say fact table

+---+------------+------+
| id|Country_Code|Amount|
+---+------------+------+
|  1|          US|   100|
|  2|          UK|   200|
|  3|          NP|   300|
|  4|         GER|   400|
+---+------------+------+



In [48]:
countries_df.show()#say dim table

+------------+--------------------+
|Country_Code|        Country_Name|
+------------+--------------------+
|          US|United States of ...|
|          UK|      United Kingdom|
|          NP|               Nepal|
|         GER|             Germany|
+------------+--------------------+



In [49]:
df_join=transactions_df.join(countries_df,transactions_df['Country_Code']==countries_df['Country_Code'],'inner')

In [54]:
df_join_optm=transactions_df.join(broadcast(countries_df),transactions_df['Country_Code']==countries_df['Country_Code'],'inner')

### SortMergeJoin Vs BroadCastJoin

In [63]:
df_join.explain(mode='formatted')

== Physical Plan ==
* SortMergeJoin Inner (9)
:- * Sort (4)
:  +- Exchange (3)
:     +- * Filter (2)
:        +- * Scan ExistingRDD (1)
+- * Sort (8)
   +- Exchange (7)
      +- * Filter (6)
         +- * Scan ExistingRDD (5)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [id#321L, Country_Code#322, Amount#323L], MapPartitionsRDD[124] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Condition : isnotnull(Country_Code#322)

(3) Exchange
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: hashpartitioning(Country_Code#322, 200), ENSURE_REQUIREMENTS, [plan_id=516]

(4) Sort [codegen id : 2]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [Country_Code#322 ASC NULLS FIRST], false, 0

(5) Scan ExistingRDD [codegen id : 3]
Output [2]: [Country_Code#324, Country_Name#325]
Arguments: 

In [62]:
df_join_optm.explain(mode='formatted')
#

== Physical Plan ==
* BroadcastHashJoin Inner BuildRight (6)
:- * Filter (2)
:  +- * Scan ExistingRDD (1)
+- BroadcastExchange (5)
   +- * Filter (4)
      +- * Scan ExistingRDD (3)


(1) Scan ExistingRDD [codegen id : 2]
Output [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [id#321L, Country_Code#322, Amount#323L], MapPartitionsRDD[124] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 2]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Condition : isnotnull(Country_Code#322)

(3) Scan ExistingRDD [codegen id : 1]
Output [2]: [Country_Code#324, Country_Name#325]
Arguments: [Country_Code#324, Country_Name#325], MapPartitionsRDD[129] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(4) Filter [codegen id : 1]
Input [2]: [Country_Code#324, Country_Name#325]
Condition : isnotnull(Country_Code#324)

(5) BroadcastExchange
Input [2]: [Country_Code#324, Coun

#### Explanation

Since optm df uses broadcast join for smaller table it does not require shuffle(exchange) and sort which are performance intensive and hence less steps incomparision to sortmergejoin the default in spark

### Spark Sql hints

In [64]:
transactions_df.createOrReplaceTempView("transactions")
countries_df.createOrReplaceTempView("countries")

In [67]:
sql_df=spark.sql(
    '''
    Select * from
    transactions as t
    Join countries as c
    On t.Country_Code=c.Country_Code
    '''
)

In [70]:
sql_df.explain(mode='formatted')

== Physical Plan ==
* SortMergeJoin Inner (9)
:- * Sort (4)
:  +- Exchange (3)
:     +- * Filter (2)
:        +- * Scan ExistingRDD (1)
+- * Sort (8)
   +- Exchange (7)
      +- * Filter (6)
         +- * Scan ExistingRDD (5)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [id#321L, Country_Code#322, Amount#323L], MapPartitionsRDD[124] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Condition : isnotnull(Country_Code#322)

(3) Exchange
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: hashpartitioning(Country_Code#322, 200), ENSURE_REQUIREMENTS, [plan_id=639]

(4) Sort [codegen id : 2]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [Country_Code#322 ASC NULLS FIRST], false, 0

(5) Scan ExistingRDD [codegen id : 3]
Output [2]: [Country_Code#324, Country_Name#325]
Arguments: 

In [83]:
sql_df_optm=spark.sql(
    '''
    Select *  /* broadcast(countries) */
    from transactions as t
    Join countries as c
    On t.Country_Code=c.Country_Code
    '''
)

In [84]:
sql_df_optm.explain(mode='formatted')

== Physical Plan ==
* SortMergeJoin Inner (9)
:- * Sort (4)
:  +- Exchange (3)
:     +- * Filter (2)
:        +- * Scan ExistingRDD (1)
+- * Sort (8)
   +- Exchange (7)
      +- * Filter (6)
         +- * Scan ExistingRDD (5)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [id#321L, Country_Code#322, Amount#323L], MapPartitionsRDD[124] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Condition : isnotnull(Country_Code#322)

(3) Exchange
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: hashpartitioning(Country_Code#322, 200), ENSURE_REQUIREMENTS, [plan_id=851]

(4) Sort [codegen id : 2]
Input [3]: [id#321L, Country_Code#322, Amount#323L]
Arguments: [Country_Code#322 ASC NULLS FIRST], false, 0

(5) Scan ExistingRDD [codegen id : 3]
Output [2]: [Country_Code#324, Country_Name#325]
Arguments: 

#### Explanation

Sql type hints enclosed within /* */ are advice to spark .It may not always take such advice as shown above

## Caching and Persistance

In [85]:
df=spark.read.format('csv')\
    .option('inferSchema',True)\
    .option('header',True)\
    .load('/content/BigMart Sales.csv')

In [87]:
df2=df.filter(col('Outlet_Type')=='Tier 1').explain(mode='formatted')

== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [12]: [Item_Identifier#408, Item_Weight#409, Item_Fat_Content#410, Item_Visibility#411, Item_Type#412, Item_MRP#413, Outlet_Identifier#414, Outlet_Establishment_Year#415, Outlet_Size#416, Outlet_Location_Type#417, Outlet_Type#418, Item_Outlet_Sales#419]
Batched: false
Location: InMemoryFileIndex [file:/content/BigMart Sales.csv]
PushedFilters: [IsNotNull(Outlet_Type), EqualTo(Outlet_Type,Tier 1)]
ReadSchema: struct<Item_Identifier:string,Item_Weight:double,Item_Fat_Content:string,Item_Visibility:double,Item_Type:string,Item_MRP:double,Outlet_Identifier:string,Outlet_Establishment_Year:int,Outlet_Size:string,Outlet_Location_Type:string,Outlet_Type:string,Item_Outlet_Sales:double>

(2) Filter [codegen id : 1]
Input [12]: [Item_Identifier#408, Item_Weight#409, Item_Fat_Content#410, Item_Visibility#411, Item_Type#412, Item_MRP#413, Outlet_Identifier#414, Outlet_Establishment_Year#415, Outlet_Size#416, Outlet_Location_

In [88]:
cached_df=spark.read.format('csv')\
    .option('inferSchema',True)\
    .option('header',True)\
    .load('/content/BigMart Sales.csv')\
    .cache()

In [89]:
df2=cached_df.filter(col('Outlet_Type')=='Tier 1').explain(mode='formatted')

== Physical Plan ==
* Filter (4)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- Scan csv  (3)


(1) InMemoryTableScan
Output [12]: [Item_Identifier#438, Item_Weight#439, Item_Fat_Content#440, Item_Visibility#441, Item_Type#442, Item_MRP#443, Outlet_Identifier#444, Outlet_Establishment_Year#445, Outlet_Size#446, Outlet_Location_Type#447, Outlet_Type#448, Item_Outlet_Sales#449]
Arguments: [Item_Identifier#438, Item_Weight#439, Item_Fat_Content#440, Item_Visibility#441, Item_Type#442, Item_MRP#443, Outlet_Identifier#444, Outlet_Establishment_Year#445, Outlet_Size#446, Outlet_Location_Type#447, Outlet_Type#448, Item_Outlet_Sales#449], [isnotnull(Outlet_Type#448), (Outlet_Type#448 = Tier 1)]

(2) InMemoryRelation
Arguments: [Item_Identifier#438, Item_Weight#439, Item_Fat_Content#440, Item_Visibility#441, Item_Type#442, Item_MRP#443, Outlet_Identifier#444, Outlet_Establishment_Year#445, Outlet_Size#446, Outlet_Location_Type#447, Outlet_Type#448, Item_Outlet_Sales#449],

### Explanation

Spark does not cache or store the df everytime.it stores in executor memory withtin spark memory pool and then allocates that memory when another action is to be performed.So after caching in explain we see 'InmemoryTableScan"

In [90]:
df.unpersist()

DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double]

### Storage Levels

Storage Levels

1.   DISK_ONLY
2.   MEMORY_ONLY
3.   MEMORY_AND_DISK

and much more

In [91]:
from pyspark.storagelevel import StorageLevel

In [93]:
df.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double]

In [94]:
df.unpersist()

DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double]